# 1. Cargamos los datos (Desde Drive .Csv anterior Challenge Parte 1)

In [77]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/TelecomX_Data_Limpio.csv')


#2. Eliminar ID (Anterior modelo)

In [78]:
df = df.drop(columns=['customerID'])


#3. Eliminar NaN en Churn

In [79]:
df = df.dropna(subset=['Churn'])


#4. Separamos X e y

In [80]:
X = df.drop(columns=['Churn'])
y = df['Churn']


#5. Split

In [81]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)


#6. Identificamos las columnas

In [82]:
num_cols = X.select_dtypes(include=['int64','float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns


#7. Creamos preprocessor



In [83]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first'), cat_cols)
    ]
)


#8. Modelos balanceados

In [84]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

log_model = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

rf_model = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight='balanced'
    ))
])


#9. Entrenamiento

In [85]:
log_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'PaperlessBilling', 'Monthly', 'Total'],
      dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(drop='first'),
                                                  Index(['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'Contract', 'PaymentMethod'],
      dtype='object'))])),
                ('model',
                 RandomForestClassifier(class_weight='balanced',
                                        n_estimators=200, random_state=42))])

#10. Función evaluar

In [86]:
from sklearn.metrics import classification_report, roc_auc_score

def evaluar(modelo):
    y_pred = modelo.predict(X_test)
    y_prob = modelo.predict_proba(X_test)[:,1]

    print(classification_report(y_test, y_pred))
    print("ROC AUC:", roc_auc_score(y_test, y_prob))


#11. Evaluamos los resultados

In [89]:
print("Logistic Regression")
evaluar(log_model)

print("Random Forest")
evaluar(rf_model)


Logistic Regression
              precision    recall  f1-score   support

         0.0       0.91      0.71      0.80      1552
         1.0       0.50      0.80      0.62       561

    accuracy                           0.73      2113
   macro avg       0.70      0.76      0.71      2113
weighted avg       0.80      0.73      0.75      2113

ROC AUC: 0.8405875002297077
Random Forest
              precision    recall  f1-score   support

         0.0       0.83      0.89      0.86      1552
         1.0       0.62      0.48      0.54       561

    accuracy                           0.78      2113
   macro avg       0.72      0.69      0.70      2113
weighted avg       0.77      0.78      0.77      2113

ROC AUC: 0.8218812595328665


#12 Interpretación estrátegica

El modelo:
*   Detecta 8 de cada 10 clientes que se van.
*   Pero también marca muchos que no se iban (precision 0.50)

#13. Análisis de la evaluación

Aunque Random Forest obtuvo mayor accuracy (0.78), la Regresión Logística presenta un recall significativamente superior para la clase churn (0.80 vs 0.48), lo cual la convierte en el modelo más adecuado para la estrategia de retención.

Dado que el objetivo de negocio es anticipar cancelaciones, priorizar la detección de clientes en riesgo es más importante que maximizar la exactitud global.

#14. Variables más importantes (Regresión Logística)

In [90]:
import numpy as np

# Obtener nombres de variables transformadas
feature_names = (
    num_cols.tolist() +
    list(log_model.named_steps['preprocess']
         .named_transformers_['cat']
         .get_feature_names_out(cat_cols))
)

# Extraer coeficientes
coef = log_model.named_steps['model'].coef_[0]

coef_df = pd.DataFrame({
    'Variable': feature_names,
    'Coeficiente': coef
})

coef_df = coef_df.sort_values(by='Coeficiente', ascending=False)

coef_df.head(10)


,Variable,Coeficiente
11,InternetService_Fiber optic,0.971981
8,Total,0.671900
28,PaymentMethod_Electronic check,0.392633
22,StreamingTV_Yes,0.301461
24,StreamingMovies_Yes,0.290531
10,MultipleLines_Yes,0.259787
6,PaperlessBilling,0.198799
1,SeniorCitizen,0.069441
0,gender,0.043351
18,DeviceProtection_Yes,-0.006011


14.1 Casos Negativos

In [91]:
coef_df.tail(10)


,Variable,Coeficiente
12,InternetService_No,-0.153571
17,DeviceProtection_No internet service,-0.153571
19,TechSupport_No internet service,-0.153571
21,StreamingTV_No internet service,-0.153571
14,OnlineSecurity_Yes,-0.246985
20,TechSupport_Yes,-0.338669
7,Monthly,-0.349629
25,Contract_One year,-0.743122
26,Contract_Two year,-1.349900
4,tenure,-1.368090
